In [14]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize

In [15]:
df = pd.read_excel("cleaneddata.xlsx")
df.columns = df.columns.str.strip()

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.sort_values(["cusip","date"])

# numeric cleanup
for col in ["spread","price","sduration","coupon","ytm"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# keep needed columns only
df = df.dropna(subset=["date","spread","price","sduration"])

# restrict period
df = df[df["date"].between("2017-01-01","2019-12-31")]
df["ym"] = df["date"].dt.to_period("M")

price_col = "price"

In [16]:
first_day = (
    df.sort_values("date")
      .groupby(["cusip","ym"])
      .first()
      .reset_index()
)

# next month price + next month label
first_day["next_price"] = first_day.groupby("cusip")[price_col].shift(-1)
first_day["next_ym"]    = first_day.groupby("cusip")["ym"].shift(-1)

# drop incomplete rows
first_day = first_day.dropna(subset=["next_price","next_ym"])

# enforce valid month pairs
first_day = first_day[
    first_day["ym"].between("2017-01","2019-12") &
    first_day["next_ym"].between("2017-01","2019-12")
]

In [17]:
first_day["price_ret"] = (
    (first_day["next_price"] - first_day[price_col]) /
     first_day[price_col]
)

# carry = coupon%/12 / price
first_day["carry"] = ((first_day["coupon"] / 100) / 12) / first_day[price_col]

# total return
first_day["ret"] = first_day["price_ret"] + first_day["carry"]

# DTS (for sorting into buckets; scaling doesn't affect order)
first_day["dts"] = first_day["spread"] * first_day["sduration"]

# wide monthly return matrix
ret_pivot = (
    first_day.pivot(index="ym", columns="cusip", values="ret")
    .sort_index()
)

In [18]:
def optimize_weights(mu, cov, w_max=0.05):
    """Maximize Sharpe ratio with weight caps (long-only)."""
    n = len(mu)
    w0 = np.ones(n) / n

    def neg_sharpe(w):
        ret = np.dot(w, mu)
        vol = np.sqrt(np.dot(w.T, np.dot(cov, w)))
        if vol <= 0:
            return 1e9
        return -(ret / vol)

    bounds = [(0, w_max)] * n
    cons = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]

    res = minimize(
        neg_sharpe, w0,
        method="SLSQP",
        bounds=bounds,
        constraints=cons,
        options={"maxiter": 500, "ftol": 1e-9}
    )

    return res.x

In [19]:
port_low  = []
port_mid  = []
port_high = []

months = sorted(first_day["ym"].unique())

for idx in range(3, len(months)):
    ym = months[idx]

    # trailing 3 months for mean/cov estimation
    window = months[idx-3:idx]
    hist = ret_pivot.loc[window]

    # current month bonds
    month_df = first_day[first_day["ym"] == ym].copy()
    month_df = month_df.sort_values("dts")
    n = len(month_df)
    if n < 6:
        continue

    # split into 3 equal DTS buckets
    k = n // 3
    low  = month_df.iloc[:k].copy()
    mid  = month_df.iloc[k:2*k].copy()
    high = month_df.iloc[2*k:3*k].copy()

    # helper
    def prepare_bucket(bucket):
        cus = list(bucket["cusip"])
        sub = hist[cus].dropna(axis=1, how="any")

        if sub.shape[1] < 2:
            return None, None, None

        cusips = list(sub.columns)
        bucket_ordered = bucket.set_index("cusip").loc[cusips].reset_index()

        mu  = sub.mean().values
        cov = np.cov(sub.T)

        return bucket_ordered, mu, cov

    # prepare each
    low_b,  mu_low,  cov_low  = prepare_bucket(low)
    mid_b,  mu_mid,  cov_mid  = prepare_bucket(mid)
    high_b, mu_high, cov_high = prepare_bucket(high)

    if low_b is None or mid_b is None or high_b is None:
        continue

    # optimize
    w_low  = optimize_weights(mu_low,  cov_low)
    w_mid  = optimize_weights(mu_mid,  cov_mid)
    w_high = optimize_weights(mu_high, cov_high)

    # realized return
    ret_low  = np.dot(w_low,  low_b["ret"].values)
    ret_mid  = np.dot(w_mid,  mid_b["ret"].values)
    ret_high = np.dot(w_high, high_b["ret"].values)

    # --- NEW: portfolio duration (for Treasury hedge)
    dur_low  = np.dot(w_low,  low_b["sduration"].values)
    dur_mid  = np.dot(w_mid,  mid_b["sduration"].values)
    dur_high = np.dot(w_high, high_b["sduration"].values)

    # append
    port_low.append({
        "month": ym,
        "ret": ret_low,
        "dur_p": dur_low,
        "cusips": list(low_b["cusip"]),
    })

    port_mid.append({
        "month": ym,
        "ret": ret_mid,
        "dur_p": dur_mid,
        "cusips": list(mid_b["cusip"]),
    })

    port_high.append({
        "month": ym,
        "ret": ret_high,
        "dur_p": dur_high,
        "cusips": list(high_b["cusip"]),
    })

In [20]:
low_opt  = pd.DataFrame(port_low)
mid_opt  = pd.DataFrame(port_mid)
high_opt = pd.DataFrame(port_high)

# cumulative returns (unhedged)
low_opt["cum_ret"]  = (1 + low_opt["ret"]).cumprod() - 1
mid_opt["cum_ret"]  = (1 + mid_opt["ret"]).cumprod() - 1
high_opt["cum_ret"] = (1 + high_opt["ret"]).cumprod() - 1

print("LOW DTS (unhedged):")
print(low_opt.tail(), "\n")

print("MID DTS (unhedged):")
print(mid_opt.tail(), "\n")

print("HIGH DTS (unhedged):")
print(high_opt.tail(), "\n")

LOW DTS (unhedged):
      month       ret   dur_p  \
27  2019-07  0.009356  4.1795   
28  2019-08  0.006965  4.1260   
29  2019-09  0.000520  4.0640   
30  2019-10 -0.004590  4.7095   
31  2019-11 -0.001116  4.2255   

                                               cusips   cum_ret  
27  [37045XBN5, 254010AC5, 38143CCX7, 06406RAJ6, 2...  0.027489  
28  [254010AC5, 37045XBN5, 38143CCX7, 06406RAJ6, 2...  0.034645  
29  [254010AC5, 38143CCX7, 06406RAJ6, 25470DAQ2, 2...  0.035184  
30  [38143CCX7, 06406RAJ6, 25470DAQ2, 25468PDK9, 1...  0.030433  
31  [38143CCX7, 06406RAJ6, 25468PDK9, 25470DAQ2, 8...  0.029282   

MID DTS (unhedged):
      month       ret    dur_p  \
27  2019-07  0.016319   9.3970   
28  2019-08  0.019504   9.0565   
29  2019-09 -0.006267   8.4050   
30  2019-10 -0.002223   9.8130   
31  2019-11  0.007451  10.1770   

                                               cusips   cum_ret  
27  [10948WAA1, 68389XAM7, 85172FAN9, 88579YAH, 44...  0.031994  
28  [85172FAN9, 68389XAM7,

In [21]:
treasury = pd.read_csv("treasury_data.csv")  # columns: ym, ret_T
treasury["ym"] = treasury["ym"].astype("period[M]")
treasury = treasury.set_index("ym").sort_index()

# Assume a constant Treasury duration (e.g. IEF ~ 7.5 years)
duration_T = 7.5


In [22]:
for df_opt in [low_opt, mid_opt, high_opt]:
    df_opt["ym"] = df_opt["month"]
    df_opt.set_index("ym", inplace=True)

# Join Treasury returns
low_opt  = low_opt.join(treasury, how="inner")
mid_opt  = mid_opt.join(treasury, how="inner")
high_opt = high_opt.join(treasury, how="inner")

# Hedge ratios: theta_T = -Dur_portfolio / Dur_Treasury
low_opt["theta_T"]  = -low_opt["dur_p"]  / duration_T
mid_opt["theta_T"]  = -mid_opt["dur_p"]  / duration_T
high_opt["theta_T"] = -high_opt["dur_p"] / duration_T

# Duration-hedged returns
low_opt["ret_hedged"]  = low_opt["ret"]  + low_opt["theta_T"] * low_opt["ret_T"]
mid_opt["ret_hedged"]  = mid_opt["ret"]  + mid_opt["theta_T"] * mid_opt["ret_T"]
high_opt["ret_hedged"] = high_opt["ret"] + high_opt["theta_T"] * high_opt["ret_T"]

# Cumulative duration-hedged returns
low_opt["cum_hedged"]  = (1 + low_opt["ret_hedged"]).cumprod() - 1
mid_opt["cum_hedged"]  = (1 + mid_opt["ret_hedged"]).cumprod() - 1
high_opt["cum_hedged"] = (1 + high_opt["ret_hedged"]).cumprod() - 1

print("LOW DTS – Duration-Hedged with Treasuries:")
print(low_opt[["month","ret","dur_p","ret_T","theta_T","ret_hedged","cum_hedged"]].tail(), "\n")

print("MID DTS – Duration-Hedged with Treasuries:")
print(mid_opt[["month","ret","dur_p","ret_T","theta_T","ret_hedged","cum_hedged"]].tail(), "\n")

print("HIGH DTS – Duration-Hedged with Treasuries:")
print(high_opt[["month","ret","dur_p","ret_T","theta_T","ret_hedged","cum_hedged"]].tail())

LOW DTS – Duration-Hedged with Treasuries:
           month       ret   dur_p     ret_T   theta_T  ret_hedged  cum_hedged
ym                                                                            
2019-07  2019-07  0.009356  4.1795  0.000594 -0.557267    0.009025   -0.021063
2019-08  2019-08  0.006965  4.1260  0.039495 -0.550133   -0.014763   -0.035515
2019-09  2019-09  0.000520  4.0640 -0.011763 -0.541867    0.006894   -0.028866
2019-10  2019-10 -0.004590  4.7095  0.002026 -0.627933   -0.005862   -0.034559
2019-11  2019-11 -0.001116  4.2255 -0.006845 -0.563400    0.002740   -0.031913 

MID DTS – Duration-Hedged with Treasuries:
           month       ret    dur_p     ret_T   theta_T  ret_hedged  \
ym                                                                    
2019-07  2019-07  0.016319   9.3970  0.000594 -1.252933    0.015575   
2019-08  2019-08  0.019504   9.0565  0.039495 -1.207533   -0.028187   
2019-09  2019-09 -0.006267   8.4050 -0.011763 -1.120667    0.006915   
2019